# مولد نمودارهای فارسی فصل پنجم

این نوت‌بوک هشت نمودار فصل پنجم را مستقیماً از داده‌های نسخه‌بندی‌شده مخزن
تولید می‌کند. هیچ مقدار آزمایشی در کد ترسیم وارد نشده است و همه مقادیر عددی
از فایل‌های CSV و JSON مستندشده خوانده و پیش از ترسیم کنترل می‌شوند.

خروجی‌ها در پوشه Draft/figures_persian به‌صورت PNG با وضوح ۳۰۰ نقطه در اینچ
و SVG برداری ذخیره می‌شوند. نمودارهای ذخیره‌شده در مخزن با فونت B Nazanin
برای متن فارسی و Times New Roman برای متن فنی انگلیسی تولید شده‌اند. هنگام
اجرای نوت‌بوک، نام فونت انتخاب‌شده به‌صراحت نمایش داده می‌شود.

In [1]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys
import warnings


required_packages = {
    "arabic_reshaper": "arabic-reshaper>=3.0.0",
    "bidi": "python-bidi>=0.6.0",
}
missing_packages = [
    package_spec
    for module_name, package_spec in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_packages]
    )

import arabic_reshaper
from bidi.algorithm import get_display
import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


def find_repository_root(start=None):
    # مخزن را بدون وابستگی به مسیر ویژه یک رایانه پیدا می‌کند.
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        marker = candidate / "reports" / "week4-week7-master-comparison.csv"
        if marker.is_file():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run the notebook from inside unlearning-thesis."
    )


REPO_ROOT = find_repository_root()
OUTPUT_DIR = REPO_ROOT / "Draft" / "figures_persian"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "identities": REPO_ROOT / "Week 2/data/synthetic_facts_v1/identities.csv",
    "facts": REPO_ROOT / "Week 2/data/synthetic_facts_v1/facts_long.csv",
    "evaluation": REPO_ROOT / "Week 2/data/synthetic_facts_v1/qa_eval_all.csv",
    "dataset_metadata": REPO_ROOT / "Week 2/data/synthetic_facts_v1/metadata.json",
    "strict_baseline": REPO_ROOT / "Week 3.5/results/qwen05_high_accuracy_baseline/metrics.json",
    "direct_ascent_samples": REPO_ROOT / "Week 4/results/gradient_ascent_unlearning_v1/results/all_before_after_results.csv",
    "direct_ascent_history": REPO_ROOT / "Week 4/results/gradient_ascent_unlearning_v1/results/unlearning_history.csv",
    "preservation_sweep": REPO_ROOT / "Week 5/results/retain_regularized_unlearning_resumable_v1/results/candidate_best_summary.csv",
    "cross_experiment": REPO_ROOT / "reports/week4-week7-master-comparison.csv",
}

missing = [
    str(path.relative_to(REPO_ROOT))
    for path in SOURCES.values()
    if not path.is_file()
]
if missing:
    raise FileNotFoundError("Missing required source files:\n" + "\n".join(missing))


def choose_font(candidates, role):
    for family in candidates:
        try:
            path = font_manager.findfont(
                FontProperties(family=family), fallback_to_default=False
            )
        except ValueError:
            continue
        return FontProperties(fname=path), family, Path(path)
    raise RuntimeError(f"No usable {role} font was found. Tried: {candidates}")


FA_FONT, FA_FONT_NAME, FA_FONT_PATH = choose_font(
    ["B Nazanin", "Vazirmatn", "Noto Naskh Arabic", "DejaVu Sans"],
    "Persian",
)
EN_FONT, EN_FONT_NAME, EN_FONT_PATH = choose_font(
    ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "English",
)

if FA_FONT_NAME != "B Nazanin":
    warnings.warn(
        f"B Nazanin is unavailable; Persian text will use {FA_FONT_NAME}. "
        "Install B Nazanin before generating the thesis-ready outputs."
    )
if EN_FONT_NAME != "Times New Roman":
    warnings.warn(
        f"Times New Roman is unavailable; English text will use {EN_FONT_NAME}. "
        "Install Times New Roman before generating the thesis-ready outputs."
    )

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": [EN_FONT_NAME, "Times New Roman", "Liberation Serif", "DejaVu Serif"],
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 9.5,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "axes.unicode_minus": False,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
        "svg.fonttype": "path",
        "mathtext.fontset": "custom",
        "mathtext.rm": EN_FONT_NAME,
        "mathtext.it": f"{EN_FONT_NAME}:italic",
        "mathtext.bf": f"{EN_FONT_NAME}:bold",
    }
)

PALETTE = {
    "navy": "#17365D",
    "blue": "#2F75B5",
    "teal": "#2A9D8F",
    "green": "#70AD47",
    "gold": "#E9C46A",
    "orange": "#F4A261",
    "red": "#C94C4C",
    "gray": "#7F8C8D",
    "light": "#EEF3F8",
}


MATPLOTLIB_NATIVE_RTL = tuple(
    int(part) for part in matplotlib.__version__.split(".")[:2]
) >= (3, 11)


def fa(text):
    # نسخه‌های جدید Matplotlib چیدمان راست‌به‌چپ و اتصال حروف را بومی انجام می‌دهند.
    if MATPLOTLIB_NATIVE_RTL:
        return str(text)
    return "\n".join(
        get_display(arabic_reshaper.reshape(line))
        for line in str(text).split("\n")
    )


def font_with_size(base_font, size, weight=None):
    result = base_font.copy()
    result.set_size(size)
    if weight is not None:
        result.set_weight(weight)
    return result


def save_figure(fig, stem):
    # یک نسخه چاپی PNG و یک نسخه برداری SVG ذخیره می‌کند.
    png_path = OUTPUT_DIR / f"{stem}.png"
    svg_path = OUTPUT_DIR / f"{stem}.svg"
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"Glyph (108|112) .* B Nazanin",
        )
        fig.savefig(png_path, dpi=300, bbox_inches="tight", pad_inches=0.12)
        fig.savefig(svg_path, format="svg", bbox_inches="tight", pad_inches=0.12)
    svg_text = svg_path.read_text(encoding="utf-8")
    normalized_svg = "\n".join(
        line.rstrip() for line in svg_text.splitlines()
    ) + "\n"
    svg_path.write_text(
        normalized_svg, encoding="utf-8", newline="\n"
    )
    plt.close(fig)
    return png_path.relative_to(REPO_ROOT), svg_path.relative_to(REPO_ROOT)


def percent_label(value):
    return f"{value:.1f}%" if abs(value - round(value)) > 1e-8 else f"{value:.0f}%"


print("Data source root: repository detected")
print("Figure output:   Draft/figures_persian")
print(f"Persian font:    {FA_FONT_NAME}")
print(f"English font:    {EN_FONT_NAME}")
print(f"Native RTL:      {MATPLOTLIB_NATIVE_RTL} (Matplotlib {matplotlib.__version__})")

Data source root: repository detected
Figure output:   Draft/figures_persian
Persian font:    B Nazanin
English font:    Times New Roman
Native RTL:      True (Matplotlib 3.11.0)


## بارگذاری و اعتبارسنجی داده‌های مستندشده آزمایش‌ها

In [2]:
identities = pd.read_csv(SOURCES["identities"])
facts = pd.read_csv(SOURCES["facts"])
evaluation = pd.read_csv(SOURCES["evaluation"])
with SOURCES["dataset_metadata"].open(encoding="utf-8") as handle:
    dataset_metadata = json.load(handle)
with SOURCES["strict_baseline"].open(encoding="utf-8") as handle:
    baseline_metrics = json.load(handle)
direct_samples = pd.read_csv(SOURCES["direct_ascent_samples"])
direct_history = pd.read_csv(SOURCES["direct_ascent_history"])
preservation_sweep = pd.read_csv(SOURCES["preservation_sweep"])
cross_experiment = pd.read_csv(SOURCES["cross_experiment"])

summary = dataset_metadata["summary"]
assert len(identities) == summary["num_identities"] == 100
assert len(facts) == summary["num_facts"] == 500
assert len(evaluation) == summary["num_eval_examples"] == 1500
assert identities["split"].value_counts().to_dict() == {"retain": 80, "forget": 20}
assert set(direct_samples["model_stage"]) == {
    "before_unlearning",
    "after_gradient_ascent",
}
assert set(direct_history["epoch"]) == set(range(1, 9))
assert len(preservation_sweep) == 9
assert cross_experiment["forget_heldout"].notna().all()
assert cross_experiment["retain_heldout"].notna().all()
assert cross_experiment["general"].notna().all()

validation_summary = pd.DataFrame(
    {
        fa("مجموعه اعتبارسنجی‌شده"): [
            fa(item) for item in [
                "هویت‌های مصنوعی",
                "واقعیت‌های مصنوعی",
                "پرسش‌های ارزیابی",
                "نامزدهای جست‌وجوی حفاظت",
                "آزمایش‌های ارزیابی کامل",
            ]
        ],
        fa("تعداد رکورد"): [
            len(identities),
            len(facts),
            len(evaluation),
            len(preservation_sweep),
            len(cross_experiment),
        ],
    }
)
validation_summary

,مجموعه اعتبارسنجی‌شده,تعداد رکورد
0,هویت‌های مصنوعی,100
1,واقعیت‌های مصنوعی,500
2,پرسش‌های ارزیابی,1500
3,نامزدهای جست‌وجوی حفاظت,9
4,آزمایش‌های ارزیابی کامل,9


## شکل ۱ ـ فرایند کامل اجرای آزمایش‌ها

In [3]:
forget_n = int((identities["split"] == "forget").sum())
retain_n = int((identities["split"] == "retain").sum())
lora_forget = 100 * baseline_metrics["lora_after_training"]["forget_all"]["contains_value"]
lora_retain = 100 * baseline_metrics["lora_after_training"]["retain_all"]["contains_value"]

pipeline = [
    ("داده مصنوعی", f"{len(identities)} هویت\n{len(facts)} واقعیت"),
    ("تفکیک هدف", f"{forget_n} هویت فراموشی\n{retain_n} هویت نگهداشت"),
    (
        "یادگیری خط مبنا",
        f"فراموشی {lora_forget:.1f}%\nنگهداشت {lora_retain:.1f}%",
    ),
    ("مداخله‌های نامزد", "صعود و حفاظت\nبه‌روزرسانی آداپتور"),
    ("کنترل تعارض", "فرافکنی و وزن‌دهی\nتطبیقی و بازگشت"),
    ("انتخاب مقید", "کنترل فراموشی، نگهداشت\nو توانایی عمومی"),
    (
        "ارزیابی کامل",
        f"{len(evaluation)} پرسش مصنوعی\nو 50 پرسش کنترل عمومی",
    ),
]

fig, ax = plt.subplots(figsize=(13.5, 8.0))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
box_h = 0.25
top_y, bottom_y = 0.56, 0.12
top_w, bottom_w = 0.23, 0.20
# مراحل ۱ تا ۳ راست‌به‌چپ در ردیف بالا و مراحل ۴ تا ۷ به‌صورت ادامه مارپیچی در ردیف پایین.
box_layout = [
    (0.73, top_y, top_w, box_h),
    (0.385, top_y, top_w, box_h),
    (0.04, top_y, top_w, box_h),
    (0.055, bottom_y, bottom_w, box_h),
    (0.285, bottom_y, bottom_w, box_h),
    (0.515, bottom_y, bottom_w, box_h),
    (0.745, bottom_y, bottom_w, box_h),
]
fills = ["#DDEBF7", "#E2F0D9", "#FFF2CC", "#FCE4D6", "#E4DFEC", "#D9EAD3", "#DDEBF7"]

for index, ((title, detail), (x0, y0, box_w, box_h), fill) in enumerate(
    zip(pipeline, box_layout, fills), start=1
):
    box = FancyBboxPatch(
        (x0, y0),
        box_w,
        box_h,
        boxstyle="round,pad=0.012,rounding_size=0.018",
        facecolor=fill,
        edgecolor=PALETTE["navy"],
        linewidth=1.5,
    )
    ax.add_patch(box)
    ax.text(
        x0 + box_w / 2,
        y0 + box_h * 0.68,
        fa(title),
        ha="center",
        va="center",
        fontproperties=font_with_size(FA_FONT, 12.5, "bold"),
    )
    ax.text(
        x0 + box_w / 2,
        y0 + box_h * 0.29,
        fa(detail),
        ha="center",
        va="center",
        linespacing=1.35,
        fontproperties=font_with_size(FA_FONT, 11),
    )
    ax.text(
        x0 + box_w - 0.012,
        y0 + box_h - 0.035,
        str(index),
        ha="center",
        va="center",
        color="white",
        fontproperties=font_with_size(EN_FONT, 8.5, "bold"),
        bbox=dict(
            boxstyle="circle,pad=0.22",
            facecolor=PALETTE["navy"],
            edgecolor="none",
        ),
    )

for source_index, target_index in zip(range(len(pipeline) - 1), range(1, len(pipeline))):
    x0, y0, box_w, box_h = box_layout[source_index]
    x1, y1, target_w, target_h = box_layout[target_index]
    if abs(y0 - y1) < 1e-9:
        if x1 < x0:
            start = (x0 - 0.006, y0 + box_h / 2)
            end = (x1 + target_w + 0.006, y1 + target_h / 2)
        else:
            start = (x0 + box_w + 0.006, y0 + box_h / 2)
            end = (x1 - 0.006, y1 + target_h / 2)
    else:
        start = (x0 + box_w / 2, y0 - 0.006)
        end = (x1 + target_w / 2, y1 + target_h + 0.006)
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            arrowstyle="-|>",
            mutation_scale=15,
            linewidth=1.4,
            color=PALETTE["gray"],
        )
    )

ax.text(
    0.5,
    0.93,
    fa("فرایند اجرای آزمایش‌های پادیادگیری"),
    ha="center",
    va="center",
    fontproperties=font_with_size(FA_FONT, 18, "bold"),
)
save_figure(fig, "01_experimental_pipeline")

(WindowsPath('Draft/figures_persian/01_experimental_pipeline.png'),
 WindowsPath('Draft/figures_persian/01_experimental_pipeline.svg'))

## شکل ۲ ـ ترکیب مجموعه‌های فراموشی و نگهداشت در سه سطح داده

In [4]:
partition_counts = pd.DataFrame(
    {
        "Forget": [
            (identities["split"] == "forget").sum(),
            (facts["split"] == "forget").sum(),
            (evaluation["split"] == "forget").sum(),
        ],
        "Retain": [
            (identities["split"] == "retain").sum(),
            (facts["split"] == "retain").sum(),
            (evaluation["split"] == "retain").sum(),
        ],
    },
    index=["هویت‌ها", "واقعیت‌ها", "پرسش‌های ارزیابی"],
)
partition_pct = partition_counts.div(partition_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10.5, 5.1))
y = np.arange(len(partition_counts))
left = np.zeros(len(partition_counts))
legend_labels = {"Forget": "فراموشی", "Retain": "نگهداشت"}
for name, color in [("Forget", PALETTE["red"]), ("Retain", PALETTE["teal"])]:
    values = partition_pct[name].to_numpy()
    bars = ax.barh(
        y,
        values,
        left=left,
        color=color,
        height=0.56,
        label=fa(legend_labels[name]),
    )
    for row_index, (bar, pct, count) in enumerate(
        zip(bars, values, partition_counts[name])
    ):
        ax.text(
            left[row_index] + pct / 2,
            bar.get_y() + bar.get_height() / 2,
            f"{int(count):,}\n({pct:.0f}%)",
            ha="center",
            va="center",
            color="white",
            fontproperties=font_with_size(EN_FONT, 10, "bold"),
        )
    left += values

ax.set_yticks(
    y,
    [fa(item) for item in partition_counts.index],
    fontproperties=font_with_size(FA_FONT, 12),
)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel(
    fa("سهم هر سطح از داده (درصد)"),
    fontproperties=font_with_size(FA_FONT, 13),
)
ax.set_title(
    fa("تقسیم‌بندی فراموشی و نگهداشت در مجموعه داده مصنوعی"),
    fontproperties=font_with_size(FA_FONT, 17, "bold"),
)
ax.legend(
    ncol=2,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.32),
    prop=font_with_size(FA_FONT, 11),
)
ax.xaxis.grid(True, color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "02_dataset_partitions")

(WindowsPath('Draft/figures_persian/02_dataset_partitions.png'),
 WindowsPath('Draft/figures_persian/02_dataset_partitions.svg'))

## شکل ۳ ـ یادگیری خط مبنای سخت‌گیرانه، پیش و پس از آموزش آداپتور

In [5]:
metric_specs = [
    ("فراموشی\nهمه پرسش‌ها", "forget_all"),
    ("نگهداشت\nهمه پرسش‌ها", "retain_all"),
    ("فراموشی\nبازنویسی جدید", "forget_heldout_paraphrases"),
    ("نگهداشت\nبازنویسی جدید", "retain_heldout_paraphrases"),
    ("فراموشی\nپرسش همسان", "forget_seen_prompts"),
    ("نگهداشت\nپرسش همسان", "retain_seen_prompts"),
]
labels = [item[0] for item in metric_specs] + ["کنترل\nعمومی"]
before = [
    100 * baseline_metrics["base_before_training"][key]["contains_value"]
    for _, key in metric_specs
]
after = [
    100 * baseline_metrics["lora_after_training"][key]["contains_value"]
    for _, key in metric_specs
]
before.append(
    baseline_metrics["general_control"][
        "base_before_training_contains_value_percentage"
    ]
)
after.append(
    baseline_metrics["general_control"][
        "lora_after_training_contains_value_percentage"
    ]
)

fig, ax = plt.subplots(figsize=(13.2, 6.4))
x = np.arange(len(labels))
width = 0.36
bars_before = ax.bar(
    x - width / 2,
    before,
    width,
    color="#A9B7C6",
    label=fa("مدل پایه پیش از آموزش"),
)
bars_after = ax.bar(
    x + width / 2,
    after,
    width,
    color=PALETTE["blue"],
    label=fa("آداپتور لورا پس از آموزش"),
)
for bars in (bars_before, bars_after):
    for bar in bars:
        value = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 1.4,
            percent_label(value),
            ha="center",
            va="bottom",
            fontproperties=font_with_size(EN_FONT, 8.8),
        )

ax.set_xticks(
    x,
    [fa(item) for item in labels],
    fontproperties=font_with_size(FA_FONT, 11),
)
ax.set_ylim(0, 112)
ax.set_ylabel(
    fa("دقت تطبیق مقدار (درصد)"),
    fontproperties=font_with_size(FA_FONT, 13),
)
ax.set_title(
    fa("خط مبنای یادگیری پردقت، پیش و پس از آموزش آداپتور لورا"),
    fontproperties=font_with_size(FA_FONT, 17, "bold"),
)
ax.legend(
    frameon=False,
    ncol=2,
    loc="upper center",
    prop=font_with_size(FA_FONT, 11),
)
ax.yaxis.grid(True, color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "03_strict_baseline_learning")

(WindowsPath('Draft/figures_persian/03_strict_baseline_learning.png'),
 WindowsPath('Draft/figures_persian/03_strict_baseline_learning.svg'))

## شکل ۴ ـ معماری چندهدفه پادیادگیری

In [6]:
fig, ax = plt.subplots(figsize=(14, 7.8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis("off")


def node(
    x,
    y,
    w,
    h,
    title,
    subtitle="",
    formula="",
    face="#FFFFFF",
    edge=PALETTE["navy"],
):
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        facecolor=face,
        edgecolor=edge,
        linewidth=1.5,
    )
    ax.add_patch(patch)
    if formula:
        ax.text(
            x + w / 2,
            y + h * 0.76,
            fa(title),
            ha="center",
            va="center",
            fontproperties=font_with_size(FA_FONT, 12, "bold"),
        )
        ax.text(
            x + w / 2,
            y + h * 0.49,
            formula,
            ha="center",
            va="center",
            fontproperties=font_with_size(EN_FONT, 10.5),
        )
        ax.text(
            x + w / 2,
            y + h * 0.20,
            fa(subtitle),
            ha="center",
            va="center",
            fontproperties=font_with_size(FA_FONT, 10.5),
        )
    else:
        ax.text(
            x + w / 2,
            y + h * 0.66,
            fa(title),
            ha="center",
            va="center",
            fontproperties=font_with_size(FA_FONT, 12, "bold"),
        )
        if subtitle:
            ax.text(
                x + w / 2,
                y + h * 0.28,
                fa(subtitle),
                ha="center",
                va="center",
                linespacing=1.3,
                fontproperties=font_with_size(FA_FONT, 10.5),
            )
    return patch


def connect(x1, y1, x2, y2, color=PALETTE["gray"], style="-|>"):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            arrowstyle=style,
            mutation_scale=15,
            linewidth=1.35,
            color=color,
            connectionstyle="arc3,rad=0",
        )
    )


node(
    0.4,
    5.45,
    2.1,
    1.05,
    "نمونه‌های فراموشی",
    "واقعیت‌های هدف",
    face="#FCE4D6",
    edge=PALETTE["red"],
)
node(
    0.4,
    2.85,
    2.1,
    1.05,
    "نمونه‌های نگهداشت",
    "واقعیت‌های مجاز",
    face="#E2F0D9",
    edge=PALETTE["teal"],
)
node(
    0.4,
    0.5,
    2.1,
    1.05,
    "مرجع ثابت",
    "مدل یادگرفته‌شده",
    face="#EDEDED",
    edge=PALETTE["gray"],
)

node(
    3.45,
    5.25,
    2.65,
    1.45,
    "هدف فراموشی",
    "گرادیان صعود مستقیم",
    formula=r"$L_f=-\mathrm{CE}_f$",
    face="#FCE4D6",
    edge=PALETTE["red"],
)
node(
    3.45,
    2.55,
    2.65,
    1.45,
    "هدف نگهداشت",
    "گرادیان حفاظت",
    formula=r"$L_r=\mathrm{CE}_r+\lambda_{KL}D_{KL}$",
    face="#E2F0D9",
    edge=PALETTE["teal"],
)

node(
    7.05,
    3.65,
    2.75,
    1.65,
    "کنترل‌گر گرادیان",
    "فرافکنی تعارض\nوزن‌دهی تطبیقی",
    face="#E4DFEC",
    edge="#8064A2",
)
node(
    10.65,
    3.65,
    2.75,
    1.65,
    "به‌روزرسانی مقید",
    "کنترل آستانه‌ها\nبازگشت هنگام نقض",
    face="#FFF2CC",
    edge="#BF9000",
)
node(
    10.65,
    1.15,
    2.75,
    1.25,
    "آداپتور لورا",
    "حالت پذیرفته‌شده مدل",
    face="#DDEBF7",
    edge=PALETTE["blue"],
)

connect(2.5, 5.98, 3.45, 5.98, PALETTE["red"])
connect(2.5, 3.38, 3.45, 3.38, PALETTE["teal"])
connect(2.5, 1.02, 3.45, 2.85, PALETTE["gray"])
connect(6.1, 5.98, 7.05, 4.88, PALETTE["red"])
connect(6.1, 3.28, 7.05, 4.08, PALETTE["teal"])
connect(9.8, 4.48, 10.65, 4.48, "#8064A2")
connect(12.02, 3.65, 12.02, 2.4, "#BF9000")
connect(10.65, 1.78, 9.3, 3.65, PALETTE["blue"])

ax.text(
    7,
    7.35,
    fa("معماری چندهدفه پادیادگیری"),
    ha="center",
    fontproperties=font_with_size(FA_FONT, 18, "bold"),
)
ax.text(
    8.55,
    6.55,
    fa("سازگارکردن گرادیان‌های متعارض\nفراموشی و نگهداشت"),
    ha="center",
    va="center",
    color="#555555",
    fontproperties=font_with_size(FA_FONT, 10.5),
)
save_figure(fig, "04_unlearning_objective_architecture")

(WindowsPath('Draft/figures_persian/04_unlearning_objective_architecture.png'),
 WindowsPath('Draft/figures_persian/04_unlearning_objective_architecture.svg'))

## شکل ۵ ـ نمای تعادل فراموشی و کارایی در ارزیابی کامل آزمایش‌ها

In [7]:
experiment_names_fa = {
    "Strict learned LoRA baseline": "خط مبنای یادگیری پردقت با آداپتور لورا",
    "Gradient ascent plus retain descent": "پادیادگیری صعود گرادیان همراه با نزول نگهداشت",
    "Retain-regularized sweep with KL preservation": "پادیادگیری منظم‌شده با نگهداشت و حفظ واگرایی",
    "Aggressive retain-regularized checkpoint": "نقطه تهاجمی پادیادگیری منظم‌شده",
    "PCGrad-style constrained gradient": "پادیادگیری با گرادیان مقید و فرافکنی تعارض",
    "Adaptive constrained unlearning": "پادیادگیری مقید تطبیقی با فشار فراموشی پویا",
    "Matched fixed-pressure control": "کنترل همسان با فشار ثابت فراموشی",
    "Rollback-constrained unlearning": "پادیادگیری مقید به بازگشت با محافظ شماره آزمایشگاه",
    "Normalized-gradient rollback": "بازگشت با گرادیان نرمال‌شده و پذیرش مشروط به پیشرفت",
}
assert set(cross_experiment["method"]) == set(experiment_names_fa)

pareto = cross_experiment.copy()
pareto["experiment_name"] = pareto["method"].map(experiment_names_fa)
pareto = pareto.sort_values(["phase", "experiment_name"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13.5, 8.8))
xmin = max(0, pareto["forget_heldout"].min() - 8)
xmax = min(100, pareto["forget_heldout"].max() + 5)
ymin = max(0, pareto["retain_heldout"].min() - 7)
ymax = 100
ax.axvspan(
    xmin,
    45,
    ymin=(82 - ymin) / (ymax - ymin),
    ymax=1,
    color="#D9EAD3",
    alpha=0.55,
    label=fa("منطقه هدف"),
)
ax.axvline(45, color=PALETTE["red"], linestyle="--", linewidth=1.3)
ax.axhline(82, color=PALETTE["teal"], linestyle="--", linewidth=1.3)

normalizer = mcolors.Normalize(
    vmin=pareto["general"].min(),
    vmax=pareto["general"].max(),
)
scatter = ax.scatter(
    pareto["forget_heldout"],
    pareto["retain_heldout"],
    c=pareto["general"],
    cmap="viridis",
    norm=normalizer,
    s=150,
    edgecolor="white",
    linewidth=1.2,
    zorder=3,
)
label_offsets = {
    1: (-8, -13),
    2: (0, 0),
    3: (0, 0),
    4: (0, 0),
    5: (0, 0),
    6: (0, 0),
    7: (0, 0),
    8: (9, 5),
    9: (-13, 5),
}
for index, row in pareto.iterrows():
    point_id = index + 1
    offset = label_offsets[point_id]
    annotation_style = dict(
        xytext=offset,
        textcoords="offset points",
        ha="center",
        va="center",
        color="white",
        fontproperties=font_with_size(EN_FONT, 8.5, "bold"),
    )
    if point_id in {1, 8, 9}:
        annotation_style["bbox"] = dict(
            boxstyle="circle,pad=0.2",
            facecolor="#333333",
            edgecolor="white",
            linewidth=0.6,
        )
    ax.annotate(
        str(point_id),
        (row["forget_heldout"], row["retain_heldout"]),
        **annotation_style,
    )

legend_lines = [
    Line2D(
        [],
        [],
        linestyle="none",
        label=fa(f"{i + 1}. {name}"),
    )
    for i, name in enumerate(pareto["experiment_name"])
]
ax.legend(
    handles=legend_lines,
    ncol=2,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.14),
    handlelength=0,
    handletextpad=0,
    columnspacing=1.5,
    prop=font_with_size(FA_FONT, 9.5),
)
cbar = fig.colorbar(scatter, ax=ax, pad=0.015)
cbar.set_label(
    fa("دقت کنترل عمومی (درصد)"),
    fontproperties=font_with_size(FA_FONT, 12),
)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel(
    fa("دقت بازنویسی فراموشی (درصد) ـ مقدار کمتر بهتر است"),
    fontproperties=font_with_size(FA_FONT, 13),
)
ax.set_ylabel(
    fa("دقت بازنویسی نگهداشت (درصد) ـ مقدار بیشتر بهتر است"),
    fontproperties=font_with_size(FA_FONT, 13),
)
ax.set_title(
    fa("تعادل فراموشی و کارایی در ارزیابی کامل آزمایش‌ها"),
    fontproperties=font_with_size(FA_FONT, 17, "bold"),
)
ax.text(
    44.2,
    82.6,
    fa("فراموشی حداکثر 45%\nنگهداشت حداقل 82%"),
    ha="right",
    va="bottom",
    color="#3D6B3D",
    fontproperties=font_with_size(FA_FONT, 10),
)
ax.grid(True, color="#EAEAEA", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "05_forgetting_utility_pareto")

(WindowsPath('Draft/figures_persian/05_forgetting_utility_pareto.png'),
 WindowsPath('Draft/figures_persian/05_forgetting_utility_pareto.svg'))

## شکل ۶ ـ حساسیت نامزدهای پادیادگیری منظم‌شده با نگهداشت

In [8]:
candidate_names_fa = {
    "c01": "نرخ یادگیری کم و وزن‌دهی متوازن",
    "c02": "نرخ یادگیری میانی و وزن‌دهی متوازن",
    "c03": "نرخ یادگیری بالا و وزن‌دهی متوازن",
    "c04": "وزن نگهداشت کمتر",
    "c05": "وزن نگهداشت بیشتر",
    "c06": "وزن واگرایی کمتر",
    "c07": "وزن واگرایی بیشتر",
    "c08": "بیشترین حفاظت",
    "c09": "تهاجمی‌ترین نامزد",
}
persian_candidate_number = {
    "c01": "۱",
    "c02": "۲",
    "c03": "۳",
    "c04": "۴",
    "c05": "۵",
    "c06": "۶",
    "c07": "۷",
    "c08": "۸",
    "c09": "۹",
}

sweep = preservation_sweep.copy()
sweep["candidate_short"] = sweep["candidate_id"].str.extract(
    r"^(c\d+)", expand=False
)
assert set(sweep["candidate_short"]) == set(candidate_names_fa)
lr_norm = mcolors.LogNorm(
    vmin=sweep["learning_rate"].min(),
    vmax=sweep["learning_rate"].max(),
)
marker_by_retain = {1.0: "o", 2.0: "s", 4.0: "D"}

fig, ax = plt.subplots(figsize=(12.8, 8.1))
for retain_weight, group in sweep.groupby("retain_weight"):
    ax.scatter(
        group["forget_heldout_selection_percentage"],
        group["retain_heldout_selection_percentage"],
        c=group["learning_rate"],
        cmap="plasma",
        norm=lr_norm,
        s=150 + 260 * group["kl_weight"],
        marker=marker_by_retain[float(retain_weight)],
        edgecolor="black",
        linewidth=0.7,
        alpha=0.9,
        zorder=3,
    )
candidate_offsets = {
    "c01": (4, -14),
    "c02": (5, -13),
    "c03": (4, -14),
    "c04": (5, 8),
    "c05": (5, 8),
    "c06": (5, -13),
    "c07": (5, 8),
    "c08": (5, -16),
    "c09": (-18, 8),
}
for _, row in sweep.iterrows():
    ax.annotate(
        persian_candidate_number[row["candidate_short"]],
        (
            row["forget_heldout_selection_percentage"],
            row["retain_heldout_selection_percentage"],
        ),
        xytext=candidate_offsets[row["candidate_short"]],
        textcoords="offset points",
        ha="center",
        va="center",
        fontproperties=font_with_size(FA_FONT, 10, "bold"),
        bbox=dict(
            boxstyle="round,pad=0.16",
            facecolor="white",
            edgecolor="#777777",
            alpha=0.82,
        ),
    )

scalar_map = plt.cm.ScalarMappable(norm=lr_norm, cmap="plasma")
scalar_map.set_array([])
cbar = fig.colorbar(scalar_map, ax=ax, pad=0.015)
cbar.set_label(
    fa("نرخ یادگیری"),
    fontproperties=font_with_size(FA_FONT, 12),
)
cbar.set_ticks(sorted(sweep["learning_rate"].unique()))
cbar.set_ticklabels(
    [f"{value:.0e}" for value in sorted(sweep["learning_rate"].unique())]
)

retain_handles = [
    Line2D(
        [0],
        [0],
        marker=marker_by_retain[value],
        color="none",
        markerfacecolor="#BFBFBF",
        markeredgecolor="black",
        markersize=9,
        label=fa(f"وزن نگهداشت = {value:g}"),
    )
    for value in sorted(marker_by_retain)
]
kl_handles = [
    plt.scatter(
        [],
        [],
        s=150 + 260 * value,
        facecolor="none",
        edgecolor="#555555",
        label=fa(f"وزن واگرایی = {value:g}"),
    )
    for value in sorted(sweep["kl_weight"].unique())
]
legend_a = ax.legend(
    handles=retain_handles,
    frameon=False,
    loc="lower right",
    title=fa("شکل نشانگر"),
    prop=font_with_size(FA_FONT, 9.5),
    title_fontproperties=font_with_size(FA_FONT, 10.5, "bold"),
)
ax.add_artist(legend_a)
ax.legend(
    handles=kl_handles,
    frameon=False,
    loc="upper left",
    title=fa("مساحت نشانگر"),
    prop=font_with_size(FA_FONT, 9.5),
    title_fontproperties=font_with_size(FA_FONT, 10.5, "bold"),
)
ax.axhline(85, color=PALETTE["teal"], linestyle="--", linewidth=1.2)
ax.text(
    sweep["forget_heldout_selection_percentage"].min(),
    85.18,
    fa("آستانه شایستگی نگهداشت (85%)"),
    ha="left",
    va="bottom",
    color=PALETTE["teal"],
    fontproperties=font_with_size(FA_FONT, 10),
)
candidate_handles = [
    Line2D(
        [],
        [],
        linestyle="none",
        label=fa(
            f"{persian_candidate_number[row.candidate_short]}. "
            f"{candidate_names_fa[row.candidate_short]}"
        ),
    )
    for row in sweep.sort_values("candidate_short").itertuples()
]
fig.legend(
    handles=candidate_handles,
    ncol=3,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.12),
    handlelength=0,
    handletextpad=0,
    columnspacing=1.5,
    prop=font_with_size(FA_FONT, 9),
)
ax.set_xlabel(
    fa("دقت انتخاب فراموشی بازنویسی‌شده (درصد) ـ مقدار کمتر بهتر است"),
    fontproperties=font_with_size(FA_FONT, 12.5),
)
ax.set_ylabel(
    fa("دقت انتخاب نگهداشت بازنویسی‌شده (درصد) ـ مقدار بیشتر بهتر است"),
    fontproperties=font_with_size(FA_FONT, 12.5),
)
ax.set_title(
    fa("حساسیت نامزدهای پادیادگیری منظم‌شده با نگهداشت"),
    fontproperties=font_with_size(FA_FONT, 17, "bold"),
)
ax.grid(True, color="#EAEAEA", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "06_hyperparameter_sensitivity")

(WindowsPath('Draft/figures_persian/06_hyperparameter_sensitivity.png'),
 WindowsPath('Draft/figures_persian/06_hyperparameter_sensitivity.svg'))

## شکل ۷ ـ مسیر پادیادگیری صعود مستقیم گرادیان

In [9]:
history = direct_history.sort_values("epoch").copy()
eligible = history["retain_eligible"].astype(str).str.lower().eq("true")
eligible_rows = history.loc[eligible]
selected_index = eligible_rows["selection_score"].idxmax()
selected = history.loc[selected_index]

fig, ax = plt.subplots(figsize=(11.5, 6.6))
ax.plot(
    history["epoch"],
    history["forget_train_percentage"],
    marker="o",
    linewidth=2.2,
    color=PALETTE["red"],
    label=fa("دقت آموزشی فراموشی"),
)
ax.plot(
    history["epoch"],
    history["retain_train_sample_percentage"],
    marker="s",
    linewidth=2.2,
    color=PALETTE["teal"],
    label=fa("دقت نمونه نگهداشت"),
)
ax.axhline(
    85,
    color="#555555",
    linestyle="--",
    linewidth=1.2,
    label=fa("آستانه نگهداشت (85%)"),
)
ax.axvspan(
    history.loc[eligible, "epoch"].min() - 0.35,
    history.loc[eligible, "epoch"].max() + 0.35,
    color="#D9EAD3",
    alpha=0.45,
    label=fa("دوره‌های واجد شرایط"),
)
ax.scatter(
    [selected["epoch"]],
    [selected["forget_train_percentage"]],
    s=220,
    facecolor="none",
    edgecolor=PALETTE["navy"],
    linewidth=2.0,
    zorder=5,
)
ax.annotate(
    fa(
        f"دوره منتخب {int(selected['epoch'])}\n"
        f"فراموشی {selected['forget_train_percentage']:.0f}%، "
        f"نگهداشت {selected['retain_train_sample_percentage']:.0f}%"
    ),
    (selected["epoch"], selected["forget_train_percentage"]),
    xytext=(24, -24),
    textcoords="offset points",
    arrowprops=dict(arrowstyle="->", color=PALETTE["navy"]),
    fontproperties=font_with_size(FA_FONT, 10.5),
)
ax.set_xticks(history["epoch"])
ax.set_ylim(0, 105)
ax.set_xlabel(
    fa("دوره آموزشی"),
    fontproperties=font_with_size(FA_FONT, 13),
)
ax.set_ylabel(
    fa("دقت (درصد)"),
    fontproperties=font_with_size(FA_FONT, 13),
)
ax.set_title(
    fa("مسیر فراموشی و نگهداشت در پادیادگیری صعود مستقیم گرادیان"),
    fontproperties=font_with_size(FA_FONT, 17, "bold"),
)
ax.legend(
    frameon=False,
    ncol=2,
    loc="lower center",
    prop=font_with_size(FA_FONT, 10.5),
)
ax.grid(True, color="#EAEAEA", linewidth=0.8)
ax.set_axisbelow(True)
save_figure(fig, "07_direct_gradient_ascent_trajectory")

(WindowsPath('Draft/figures_persian/07_direct_gradient_ascent_trajectory.png'),
 WindowsPath('Draft/figures_persian/07_direct_gradient_ascent_trajectory.svg'))

## شکل ۸ ـ اثر نوع واقعیت در پادیادگیری صعود مستقیم گرادیان

In [10]:
category = (
    direct_samples.loc[
        direct_samples["eval_split"].isin(["forget", "retain"])
    ]
    .groupby(
        ["eval_split", "category", "model_stage"],
        as_index=False,
    )["contains_value"]
    .mean()
)
category["accuracy"] = 100 * category["contains_value"]
category_order = [
    "access_phrase",
    "favorite_city",
    "lab_number",
    "research_topic",
    "secret_code",
]
category_labels = {
    "access_phrase": "عبارت دسترسی",
    "favorite_city": "شهر محبوب",
    "lab_number": "شماره آزمایشگاه",
    "research_topic": "موضوع پژوهش",
    "secret_code": "کد محرمانه",
}

fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.7), sharey=True)
panel_specs = [
    (
        "forget",
        "هویت‌های فراموشی ـ مقدار کمتر پس از مداخله بهتر است",
        PALETTE["red"],
    ),
    (
        "retain",
        "هویت‌های نگهداشت ـ مقدار بیشتر پس از مداخله بهتر است",
        PALETTE["teal"],
    ),
]
y = np.arange(len(category_order))

for ax, (split_name, title, after_color) in zip(axes, panel_specs):
    subset = category.loc[category["eval_split"] == split_name]
    pivot = (
        subset.pivot(
            index="category",
            columns="model_stage",
            values="accuracy",
        )
        .reindex(category_order)
    )
    before_values = pivot["before_unlearning"].to_numpy()
    after_values = pivot["after_gradient_ascent"].to_numpy()
    for yi, before_value, after_value in zip(y, before_values, after_values):
        ax.plot(
            [before_value, after_value],
            [yi, yi],
            color="#B8B8B8",
            linewidth=3,
            zorder=1,
        )
    ax.scatter(
        before_values,
        y,
        s=90,
        color=PALETTE["navy"],
        label=fa("پیش از پادیادگیری"),
        zorder=3,
    )
    ax.scatter(
        after_values,
        y,
        s=90,
        color=after_color,
        label=fa("پس از صعود گرادیان"),
        zorder=3,
    )
    for yi, before_value, after_value in zip(y, before_values, after_values):
        ax.text(
            before_value,
            yi - 0.22,
            percent_label(before_value),
            ha="center",
            va="bottom",
            color=PALETTE["navy"],
            fontproperties=font_with_size(EN_FONT, 8.5),
        )
        ax.text(
            after_value,
            yi + 0.22,
            percent_label(after_value),
            ha="center",
            va="top",
            color=after_color,
            fontproperties=font_with_size(EN_FONT, 8.5),
        )
    ax.set_xlim(0, 105)
    ax.set_xlabel(
        fa("دقت تطبیق مقدار (درصد)"),
        fontproperties=font_with_size(FA_FONT, 12),
    )
    ax.set_title(
        fa(title),
        fontproperties=font_with_size(FA_FONT, 13),
    )
    ax.xaxis.grid(True, color="#EAEAEA", linewidth=0.8)
    ax.set_axisbelow(True)

axes[0].set_yticks(
    y,
    [fa(category_labels[item]) for item in category_order],
    fontproperties=font_with_size(FA_FONT, 11.5),
)
axes[0].invert_yaxis()
category_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        color=PALETTE["navy"],
        markersize=9,
        label=fa("پیش از پادیادگیری"),
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        color=PALETTE["red"],
        markersize=9,
        label=fa("پس از صعود گرادیان: هویت‌های فراموشی"),
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        color=PALETTE["teal"],
        markersize=9,
        label=fa("پس از صعود گرادیان: هویت‌های نگهداشت"),
    ),
]
fig.legend(
    handles=category_handles,
    frameon=False,
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.01),
    prop=font_with_size(FA_FONT, 10),
)
fig.suptitle(
    fa("اثر نوع واقعیت در پادیادگیری صعود مستقیم گرادیان"),
    fontproperties=font_with_size(FA_FONT, 17, "bold"),
    y=1.01,
)
fig.tight_layout(rect=(0, 0.07, 1, 0.98))
save_figure(fig, "08_fact_category_effects")

(WindowsPath('Draft/figures_persian/08_fact_category_effects.png'),
 WindowsPath('Draft/figures_persian/08_fact_category_effects.svg'))

## فایل‌های تولیدشده

In [11]:
generated = sorted(
    path.relative_to(REPO_ROOT)
    for path in OUTPUT_DIR.glob("*.*")
)
assert len(generated) == 16, (
    f"Expected 16 output files, found {len(generated)}"
)
pd.DataFrame(
    {
        fa("فایل تولیدشده"): [str(path) for path in generated],
    }
)

,فایل تولیدشده
0,Draft\figures_persian\01_experimental_pipeline...
1,Draft\figures_persian\01_experimental_pipeline...
2,Draft\figures_persian\02_dataset_partitions.png
3,Draft\figures_persian\02_dataset_partitions.svg
4,Draft\figures_persian\03_strict_baseline_learn...
5,Draft\figures_persian\03_strict_baseline_learn...
6,Draft\figures_persian\04_unlearning_objective_...
7,Draft\figures_persian\04_unlearning_objective_...
8,Draft\figures_persian\05_forgetting_utility_pa...
9,Draft\figures_persian\05_forgetting_utility_pa...
